## Data Generation File

### Hidden Anomalies You Just Built
This script deploys two major corporate inefficiencies into the data infrastructure.

### The Regional Structural Leadership Crisis (CH_05):
Volunteers assigned to Chapter 5 (located in the North) have their baseline retention probability dropped to 35%, and their average event attendance cut in half. 

### The Day-45 Onboarding Bottleneck:
For any volunteer programmatically flagged to churn, their expected_last_active date is hard-capped between 14 and 75 days. Furthermore, their logged engagement_score is explicitly programmed to decay down into the 3–6 range during their final three weeks of life.


In [6]:
import pandas as pd
import numpy as np
from faker import Faker
import datetime
import os

# Initialize components and set seed for reproducible "business signals"
fake = Faker()
np.random.seed(42)
Faker.seed(42)

os.makedirs('data', exist_ok=True)

# CONFIGURATION
NUM_VOLUNTEERS = 1200
START_DATE = datetime.date(2025, 1, 1)
END_DATE = datetime.date(2026, 5, 1)

print("Starting production data simulation...")

# ==========================================
# 1. GENERATE DIM_CHAPTERS
# ==========================================
chapters_data = [
    {"chapter_id": "CH_01", "region": "North", "chapter_size": "Large"},
    {"chapter_id": "CH_02", "region": "South", "chapter_size": "Medium"},
    {"chapter_id": "CH_03", "region": "East", "chapter_size": "Small"},
    {"chapter_id": "CH_04", "region": "West", "chapter_size": "Large"},
    {"chapter_id": "CH_05", "region": "North", "chapter_size": "Small"} # Hidden Churn Target!
]
df_chapters = pd.DataFrame(chapters_data)

# ==========================================
# 2. GENERATE DIM_EVENTS
# ==========================================
event_types = ['Mentorship Session', 'Community Outreach', 'Operational Support', 'Annual Fundraiser']
events_list = []
current_date = START_DATE
event_id_counter = 100

while current_date <= END_DATE:
    # Schedule 2-4 events per week across different chapters
    if current_date.weekday() in [2, 5]:  # Wednesdays and Saturdays
        for ch in df_chapters['chapter_id'].unique():
            if np.random.rand() > 0.3:  # 70% chance an event occurs
                events_list.append({
                    "event_id": f"EVNT_{event_id_counter}",
                    "event_type": np.random.choice(event_types, p=[0.4, 0.3, 0.2, 0.1]),
                    "date": current_date,
                    "location": f"Site_{ch[-2:]}"
                })
                event_id_counter += 1
    current_date += datetime.timedelta(days=1)

df_events = pd.DataFrame(events_list)

# ==========================================
# 3. GENERATE DIM_VOLUNTEERS (With Seeded Anomalies)
# ==========================================
volunteers_list = []

for i in range(NUM_VOLUNTEERS):
    v_id = f"VOL_{10000 + i}"
    join_date = fake.date_between_dates(date_start=START_DATE, date_end=END_DATE - datetime.timedelta(days=60))
    chapter_row = df_chapters.sample(1).iloc[0]
    ch_id = chapter_row['chapter_id']
    
    # SEEDING THE BUSINESS LESSONS (Operational Flaws)
    # Flaw 1: Summer cohorts drop off due to poor seasonal onboarding
    is_summer_join = join_date.month in [6, 7, 8]
    # Flaw 2: Chapter 5 (Small/North) has severe structural management churn
    is_problem_chapter = (ch_id == "CH_05")
    
    # Calculate baseline user retention probability
    if is_problem_chapter:
        retention_prob = 0.35  # Severe churn risk
    elif is_summer_join:
        retention_prob = 0.55  # Medium churn risk
    else:
        retention_prob = 0.88  # Solid baseline engagement
        
    is_retained = np.random.choice([True, False], p=[retention_prob, 1 - retention_prob])
    
    # Determine how long they stay active before disengaging
    if not is_retained:
        tenure_days = np.random.randint(14, 75)  # Drop off early (the 45-day bottleneck)
    else:
        tenure_days = (END_DATE - join_date).days
        
    volunteers_list.append({
        "volunteer_id": v_id,
        "age_group": np.random.choice(['18-22', '23-30', '31-45', '46+'], p=[0.5, 0.2, 0.15, 0.15]),
        "region": chapter_row['region'],
        "join_date": join_date,
        "chapter_id": ch_id,
        "expected_last_active": join_date + datetime.timedelta(days=int(tenure_days))
    })

df_volunteers = pd.DataFrame(volunteers_list)

# ==========================================
# 4. GENERATE FACT_VOLUNTEER_ACTIVITY & FACT_DONATIONS
# ==========================================
activity_list = []
donations_list = []
donation_id_counter = 50000

for idx, vol in df_volunteers.iterrows():
    v_id = vol['volunteer_id']
    j_date = vol['join_date']
    l_active = vol['expected_last_active']
    ch_id = vol['chapter_id']
    
    # Find valid scheduled events matching their operational window
    valid_events = df_events[(df_events['date'] >= j_date) & (df_events['date'] <= l_active)]
    
    if len(valid_events) == 0:
        continue
        
    # Pick a random sample of events they attended based on a momentum trend
    attendance_rate = 0.6 if ch_id != "CH_05" else 0.3
    num_attended = max(1, int(len(valid_events) * attendance_rate))
    attended_events = valid_events.sample(min(num_attended, len(valid_events))).sort_values(by='date')
    
    # Generate longitudinal logs
    event_sequence = 0
    for _, ev in attended_events.iterrows():
        event_sequence += 1
        
        # Seeding a behavior trend: Closeness to churn correlates with dropping scores
        days_until_churn = (l_active - ev['date']).days
        if days_until_churn < 21 and l_active < (END_DATE - datetime.timedelta(days=15)):
            base_score = np.random.choice([3, 4, 5, 6], p=[0.3, 0.4, 0.2, 0.1]) # Dissatisfied drop-off
        else:
            base_score = np.random.choice([7, 8, 9, 10], p=[0.1, 0.3, 0.4, 0.2])
            
        activity_list.append({
            "volunteer_id": v_id,
            "event_id": ev['event_id'],
            "date": ev['date'],
            "hours_served": round(np.random.normal(3.5, 0.8), 1),
            "engagement_score": int(base_score)
        })
        
        # Secondary Business Insight: Highly engaged volunteers occasionally donate financial capital
        if base_score >= 8 and np.random.rand() > 0.85:
            donations_list.append({
                "donation_id": f"DON_{donation_id_counter}",
                "volunteer_id": v_id,
                "amount": float(np.random.choice([10.00, 25.00, 50.00, 100.00], p=[0.4, 0.3, 0.2, 0.1])),
                "date": ev['date'] + datetime.timedelta(days=1)
            })
            donation_id_counter += 1

df_activity = pd.DataFrame(activity_list)
df_donations = pd.DataFrame(donations_list)

# Clean helper column before save
df_volunteers = df_volunteers.drop(columns=['expected_last_active'])

# ==========================================
# 5. SAVE RAW FILES TO TARGET DIRECTORY
# ==========================================
df_chapters.to_csv('../data/dim_chapters.csv', index=False)
df_events.to_csv('../data/dim_events.csv', index=False)
df_volunteers.to_csv('../data/dim_volunteers.csv', index=False)
df_activity.to_csv('../data/fact_volunteer_activity.csv', index=False)
df_donations.to_csv('../data/fact_donations.csv', index=False)
print(f"Success! Data output generated inside /data directory:")
print(f" -> dim_chapters.csv:  {df_chapters.shape[0]} records")
print(f" -> dim_events.csv:    {df_events.shape[0]} records")
print(f" -> dim_volunteers.csv:{df_volunteers.shape[0]} records")
print(f" -> fact_activity.csv: {df_activity.shape[0]} records")
print(f" -> fact_donations.csv:{df_donations.shape[0]} records")

Starting production data simulation...
Success! Data output generated inside /data directory:
 -> dim_chapters.csv:  5 records
 -> dim_events.csv:    469 records
 -> dim_volunteers.csv:1200 records
 -> fact_activity.csv: 137857 records
 -> fact_donations.csv:18445 records
